In [1]:
import pandas as pd
import json
import unicodedata
import difflib

# 1. Load the data
with open('players.json', 'r', encoding='utf-8') as f:
    df_players = pd.DataFrame(json.load(f))
    
with open('players_additional.json', 'r', encoding='utf-8') as f:
    df_additional = pd.DataFrame(json.load(f))

# 2. Function to clean and normalize names
def clean_name(name):
    if not isinstance(name, str): return ""
    
    name_folded = name.casefold().strip()
    return unicodedata.normalize('NFKD', name_folded).encode('ASCII', 'ignore').decode('utf-8')

# Apply the cleaning function to create joining keys
df_players['join_name'] = df_players['player_name'].apply(clean_name)
df_additional['join_name'] = df_additional['name'].apply(clean_name)

# 3. Fuzzy matching function for the remaining aliases
def fuzzy_match(name, choices):
    for choice in choices:
        if name in choice or choice in name:
            return choice
            
    matches = difflib.get_close_matches(name, choices, n=1, cutoff=0.7)
    return matches[0] if matches else name

# 4. Map the players to the closest match in the additional dataset
additional_unmatched = df_additional['join_name'].tolist()
df_players['join_name_fuzzy'] = df_players['join_name'].apply(lambda x: fuzzy_match(x, additional_unmatched))

# 5. Perform the Left Join
df_final = df_players.merge(
    df_additional, 
    left_on='join_name_fuzzy', 
    right_on='join_name', 
    how='left',
    suffixes=('', '_add') 
)

# 6. Clean up the temporary merge columns
cols_to_drop = [col for col in df_final.columns if 'join_name' in col]
df_final = df_final.drop(columns=cols_to_drop)

# 7. Save as JSON
output_filename = 'joined_players_data.json'
df_final.to_json(output_filename, orient='records', indent=4, force_ascii=False)

print(f"Successfully saved merged data to {output_filename}")

Successfully saved merged data to joined_players_data.json


In [4]:
import pandas as pd
import json
import unicodedata
import difflib


# -----------------------------
# COMMON HELPERS
# -----------------------------

def clean_name(name):
    if not isinstance(name, str):
        return ""

    name_folded = name.casefold().strip()

    return unicodedata.normalize(
        "NFKD",
        name_folded
    ).encode(
        "ASCII",
        "ignore"
    ).decode("utf-8")


def fuzzy_match(name, choices):

    for choice in choices:
        if name in choice or choice in name:
            return choice

    matches = difflib.get_close_matches(
        name,
        choices,
        n=1,
        cutoff=0.7
    )

    return matches[0] if matches else name


# =====================================================
# 1. JOIN PLAYERS
# =====================================================

def join_players(
    players_file="players.json",
    additional_file="players_additional_new.json",
    output_file="new_players.json"
):

    df_players = pd.read_json(players_file)
    df_add = pd.read_json(additional_file)

    # create keys
    df_players["join_name"] = df_players["player_name"].apply(clean_name)
    df_add["join_name"] = df_add["name"].apply(clean_name)

    additional_names = df_add["join_name"].tolist()

    df_players["join_name_fuzzy"] = df_players["join_name"].apply(
        lambda x: fuzzy_match(x, additional_names)
    )

    df_final = df_players.merge(
        df_add,
        left_on="join_name_fuzzy",
        right_on="join_name",
        how="left",
        suffixes=("", "_add")
    )

    df_final = df_final.drop(
        columns=[c for c in df_final.columns if "join_name" in c]
    )

    df_final.to_json(
        output_file,
        orient="records",
        indent=4,
        force_ascii=False
    )

    print("Players joined →", output_file)



# =====================================================
# 2. JOIN TEAMS
# =====================================================

def join_teams(
    summary_file="team_summary.json",
    badges_file="teams_badges.json",
    output_file="new_team_summary.json"
):

    df_summary = pd.read_json(summary_file)
    df_badges = pd.read_json(badges_file)

    # ---- create keys ----
    
    print("Summary columns:", df_summary.columns.tolist())
    print("Badges columns:", df_badges.columns.tolist())

    team_col_summary = next((col for col in df_summary.columns if "team" in col.lower()), None)
    team_col_badges = next((col for col in df_badges.columns if "team" in col.lower()), None)
    
    if not team_col_summary or not team_col_badges:
        raise ValueError(f"Could not find team column. Summary: {team_col_summary}, Badges: {team_col_badges}")

    df_summary["join_name"] = df_summary[team_col_summary].apply(clean_name)
    df_badges["join_name"] = df_badges[team_col_badges].apply(clean_name)

    badge_names = df_badges["join_name"].tolist()

    df_summary["join_name_fuzzy"] = df_summary["join_name"].apply(
        lambda x: fuzzy_match(x, badge_names)
    )

    # ---- merge ----

    df_final = df_summary.merge(
        df_badges,
        left_on="join_name_fuzzy",
        right_on="join_name",
        how="left",
        suffixes=("", "_badge")
    )

    df_final = df_final.drop(
        columns=[c for c in df_final.columns if "join_name" in c]
    )

    df_final.to_json(
        output_file,
        orient="records",
        indent=4,
        force_ascii=False
    )

    print("Teams joined →", output_file)



# =====================================================
# RUN
# =====================================================

if __name__ == "__main__":

    join_players()

    join_teams()

Players joined → new_players.json
Summary columns: ['rank', 'team_name', 'matches_played', 'wins', 'draws', 'losses', 'points', 'goals_for', 'goals_against', 'goal_difference', 'form_array', 'next_opponent', 'next_home_away']
Badges columns: ['team', 'badge_url']
Teams joined → new_team_summary.json
